<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">02 Data Communication</h1>
<hr>


## **9. Shannon Capacity**

<div class="alert alert-light">

  **Name:** (add name here)
  
  **Matriculation Number:** (add matriculation number here)
  
</div>



### 9.1 Introduction: How Much Information Can a Channel Carry?

Notebook [02.8 Radio Spectrum](02.8_Radio_Spectrum.ipynb) showed how more bits per symbol can reduce the bandwidth needed for a fixed bit rate. It also raised a limitation: at the same average signal power, larger constellations become harder to distinguish in noise. This leads to a fundamental question: can increasingly sophisticated modulation and coding keep raising the reliable data rate, or is there an ultimate limit?

**Channel capacity** is the highest information rate that a channel can support with arbitrarily small error probability, given its constraints. This notebook examines the capacity of a band-limited channel with additive white Gaussian noise (AWGN). Its capacity depends on bandwidth and received signal-to-noise ratio (SNR), rather than on a particular modulation scheme.

The experiments first revisit the effect of noise on constellation points, then compare an ideal capacity limit with a simplified model of practical rate selection. The final section connects bandwidth to the thermal-noise relationship $P_n=kTB$ from notebook 02.6. Together, these ideas explain why spectrum, received power, and reliability must be considered jointly.


### 9.2 From Constellation Size to Reliable Information Rate

A modulation symbol selects one of $M$ possible signal states and can represent $\log_2(M)$ bits. Increasing the number of states raises the number of bits per symbol, but does not by itself guarantee that those bits can be recovered. At a fixed average signal power, a denser constellation leaves less separation between neighbouring states.

| Modulation | Number of states $M$ | Bits per symbol |
|---|---|---|
| BPSK | 2 | 1 |
| QPSK | 4 | 2 |
| 16-QAM | 16 | 4 |
| 64-QAM | 64 | 6 |
| 256-QAM | 256 | 8 |

The modulation bit rate is $R_b=R_s\log_2(M)$, where $R_s$ is the symbol rate. Its spectral efficiency is therefore $R_b/B=(R_s/B)\log_2(M)$. Bits per symbol and bit/s/Hz have the same numerical value only under the additional assumption $R_s/B=1$. This idealized assumption is used in the rate-selection example below; it differs from the first-null bandwidth measure used for the rectangular pulses in notebook 02.8.

Coding adds a further distinction: some transmitted bits provide redundancy rather than new information. With channel-code rate $R_c$, the information rate is $R_cR_s\log_2(M)$ before accounting for other overhead. Shannon capacity limits reliable information transfer, not the raw count of coded bits transmitted each second.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider
from IPython.display import display

schemes = ["BPSK", "QPSK", "16-QAM", "64-QAM", "256-QAM"]
bits_per_symbol = np.array([1, 2, 4, 6, 8])

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(schemes, bits_per_symbol, color="tab:blue")
ax.set(ylabel="Bits per symbol", title="More signal states carry more bits per symbol")
ax.grid(True, axis="y", alpha=0.25)
fig.tight_layout()
plt.show()


The following plots compare QPSK and 16-QAM at the same nominal average symbol energy. Both use the same noise level and the same axis limits. Crosses mark the ideal symbols, and the surrounding points show noisy observations. Lowering the SNR spreads each cloud, making a received point more likely to be assigned to the wrong symbol.

The symbol and noise samples stay fixed while the slider changes the noise amplitude, so differences between settings are not caused by drawing a new random sample. Here, the slider controls symbol energy divided by complex noise variance, interpreted as channel SNR under the idealized $R_s/B=1$ model.


In [ ]:
# Fix the transmitted symbols and noise samples for repeatable comparisons.
constellation_rng = np.random.default_rng(9)
qpsk_ideal = np.array([-1-1j, -1+1j, 1-1j, 1+1j]) / np.sqrt(2)
qam_levels = np.array([-3, -1, 1, 3])
qam16_ideal = (qam_levels[:, None] + 1j * qam_levels[None, :]).ravel() / np.sqrt(10)
constellation_examples = [
    ("QPSK", qpsk_ideal, constellation_rng.choice(qpsk_ideal, 2000)),
    ("16-QAM", qam16_ideal, constellation_rng.choice(qam16_ideal, 2000)),
]
unit_noise = (constellation_rng.standard_normal(2000)
              + 1j * constellation_rng.standard_normal(2000)) / np.sqrt(2)
constellation_limit = 1.15 * (np.max(np.abs(qam16_ideal)) + np.max(np.abs(unit_noise)))

def make_constellation_figure(snr_db):
    snr_linear = 10 ** (snr_db / 10)
    figure, axes = plt.subplots(1, 2, figsize=(10, 4.6), sharex=True, sharey=True)
    for axis, (label, ideal, transmitted) in zip(axes, constellation_examples):
        received = transmitted + unit_noise / np.sqrt(snr_linear)
        axis.scatter(received.real, received.imag, s=7, alpha=0.4,
                     color="tab:blue", label="Received symbols")
        axis.scatter(ideal.real, ideal.imag, s=65, marker="x", linewidths=1.7,
                     color="tab:orange", label="Ideal symbols", zorder=3)
        axis.set(title=label, xlabel="In-phase (I)", ylabel="Quadrature (Q)",
                 xlim=(-constellation_limit, constellation_limit),
                 ylim=(-constellation_limit, constellation_limit))
        axis.set_aspect("equal", adjustable="box")
        axis.grid(alpha=0.2)
        axis.legend(fontsize=8, loc="upper right")
    figure.suptitle(f"Equal nominal symbol energy and noise level: SNR = {snr_db:.0f} dB")
    figure.tight_layout()
    return figure

constellation_slider = FloatSlider(
    min=0, max=25, step=1, value=15, description="SNR (dB)", continuous_update=False,
)
display(constellation_slider)
constellation_figure = make_constellation_figure(constellation_slider.value)
plt.close(constellation_figure)
constellation_handle = display(constellation_figure, display_id=True)

def update_constellations(change=None):
    figure = make_constellation_figure(constellation_slider.value)
    plt.close(figure)
    constellation_handle.update(figure)

constellation_slider.observe(update_constellations, names="value")


At high SNR, most received points remain close to their ideal symbols. As the SNR decreases, the clouds spread into neighbouring decision regions. The more closely spaced 16-QAM symbols become difficult to distinguish sooner than the QPSK symbols at the same noise level.

Visual overlap is an illustration rather than a precise reliability threshold. Gaussian noise has unbounded tails, so even well-separated clouds can produce occasional errors. Coding can reduce the rate of information errors, but it cannot make an information rate above channel capacity arbitrarily reliable. The next section expresses that limit mathematically.


### 9.3 Shannon Capacity and Its Practical Meaning

#### 9.3.1 Capacity of a Band-Limited AWGN Channel

For a channel with bandwidth $B$, average received signal power $P_s$, and additive white Gaussian noise power $P_n$ within that bandwidth, the Shannon–Hartley expression is

$$
C=B\log_2\!\left(1+\frac{P_s}{P_n}\right)
=B\log_2(1+\mathrm{SNR}).
$$

Capacity $C$ is measured in bit/s and bandwidth $B$ in Hertz. The formula requires a **linear power ratio**: a value of 10 dB corresponds to $\mathrm{SNR}=10^{10/10}=10$. More generally,

$$
\mathrm{SNR}=10^{\mathrm{SNR}_{\mathrm{dB}}/10}.
$$

For information rates below $C$, suitable coding can make the error probability arbitrarily small as coding length and decoding capability increase. Rates above $C$ cannot achieve that goal under the same channel assumptions. This is an asymptotic limit, not a promise of error-free reception with any particular modulation, code, or finite packet. The underlying model assumes an average-power constraint and allows an optimized input distribution; it is not restricted to a fixed QAM constellation. See [MIT's notes on AWGN channel capacity](https://ocw.mit.edu/courses/16-36-communication-systems-engineering-spring-2009/a49d9e2954b440def3794fe73e79756c_MIT16_36s09_lec13_14.pdf).

Dividing capacity by bandwidth gives the maximum spectral efficiency under this model:

$$
\frac{C}{B}=\log_2(1+\mathrm{SNR})\qquad[\mathrm{bit/s/Hz}].
$$

At 0 dB SNR, signal and noise powers are equal and the capacity is 1 bit/s/Hz. The graph below shows that increasing SNR raises this limit. Unlike the discrete constellation sizes, the capacity curve is continuous.


In [ ]:
snr_db = np.linspace(-10, 30, 400)
snr_lin = 10**(snr_db/10)
cap_per_hz = np.log2(1 + snr_lin)

plt.figure(figsize=(6,4))
plt.plot(snr_db, cap_per_hz, label="Shannon Capacity")
plt.xlabel("SNR (dB)")
plt.ylabel("Capacity per Hz (bits/s/Hz)")
plt.title("Shannon Capacity vs SNR")
plt.grid(True)
plt.legend()
plt.show()


#### 9.3.2 Capacity and an Illustrative Rate-Selection Model

A practical system selects from a finite set of modulation and coding modes. Each mode needs sufficiently good channel conditions to meet a target error rate. The resulting information rate may rise in steps as better modes become usable, while the Shannon capacity curve provides a smooth upper limit for the assumed AWGN channel.

The following graph illustrates this distinction using a simplified **SNR-gap model**. A gap $G$ in dB corresponds to a factor $\Gamma=10^{G/10}$ and defines the reduced-rate curve

$$
\eta_{\mathrm{gap}}=\log_2\!\left(1+\frac{\mathrm{SNR}}{\Gamma}\right).
$$

The staircase selects the largest level from 1, 2, 4, 6, and 8 bit/s/Hz that does not exceed this curve. These levels use the idealized one-symbol/s/Hz assumption from Section 9.2 and serve as candidate rate levels. They are **not measured achievable rates or guaranteed switching thresholds** for uncoded BPSK or QAM. Actual thresholds depend on coding, target error rate, receiver implementation, and overhead. A zero staircase value means none of the listed levels is selected; it does not mean the channel capacity is zero.

Move the gap slider to see how a larger assumed penalty shifts the candidate-rate transitions to higher SNR. The physical channel capacity stays unchanged. This SNR gap represents a simplified performance penalty and is distinct from the fading reserve introduced in notebook 02.7.


In [ ]:
def capacity_per_hz(snr_db):
    return np.log2(1 + 10 ** (np.asarray(snr_db) / 10))

def illustrative_rates(snr_db, gap_db):
    reduced_rate = np.log2(1 + 10 ** ((np.asarray(snr_db) - gap_db) / 10))
    candidate_levels = np.array([1, 2, 4, 6, 8])
    staircase = np.max(
        np.where(candidate_levels[:, None] <= reduced_rate,
                 candidate_levels[:, None], 0), axis=0,
    )
    return reduced_rate, staircase

def make_capacity_figure(gap_db):
    snr_values = np.linspace(-5, 30, 701)
    reduced_rate, staircase = illustrative_rates(snr_values, gap_db)
    figure, axis = plt.subplots(figsize=(9, 4.8))
    axis.plot(snr_values, capacity_per_hz(snr_values),
              label="AWGN channel capacity", color="tab:blue")
    axis.plot(snr_values, reduced_rate, linestyle="--", color="tab:orange",
              label=f"Illustrative rate curve with {gap_db:.1f} dB gap")
    axis.step(snr_values, staircase, where="post", color="tab:green",
              label="Candidate rate levels (illustrative)")
    axis.set(xlabel="SNR (dB)", ylabel="Rate per bandwidth (bit/s/Hz)",
             title="Channel capacity and a simplified rate-selection model",
             ylim=(0, 10.5))
    axis.grid(alpha=0.25)
    axis.legend(fontsize=9)
    figure.tight_layout()
    return figure

gap_slider = FloatSlider(
    value=3, min=0, max=6, step=0.5, description="SNR gap (dB)", continuous_update=False,
)
display(gap_slider)
capacity_figure = make_capacity_figure(gap_slider.value)
plt.close(capacity_figure)
capacity_handle = display(capacity_figure, display_id=True)

def update_capacity_plot(change=None):
    figure = make_capacity_figure(gap_slider.value)
    plt.close(figure)
    capacity_handle.update(figure)

gap_slider.observe(update_capacity_plot, names="value")


#### 9.3.3 Bandwidth, Power, and Thermal Noise

The capacity formula appears to make bandwidth and SNR independent choices, but a physical receiver usually couples them. If thermal noise dominates and temperature stays fixed, notebook 02.6 gives $P_n=kTB$, where $k$ is Boltzmann's constant and $T$ is the noise-source temperature in kelvin. Under this simplified noise model,

$$
C(B)=B\log_2\!\left(1+\frac{P_s}{kTB}\right).
$$

Doubling bandwidth while holding SNR fixed doubles capacity. Maintaining that SNR, however, requires twice the received signal power because the wider bandwidth admits twice as much thermal noise. With received signal power fixed instead, doubling bandwidth halves SNR. Capacity still increases under this model, but by less than a factor of two. This assumes the full signal power remains inside the receiver bandwidth in both cases.

Doubling received signal power at fixed bandwidth doubles SNR, an increase of about 3 dB. Capacity grows logarithmically with this power increase rather than proportionally. In all comparisons, convert dB to a linear ratio before using the logarithm; doubling power does not mean doubling the numerical SNR value in dB.

These distinctions connect capacity to the spectrum discussion. A wider allocation can support more information, but also collects more noise. Higher spectral efficiency reduces the bandwidth needed for a given information rate only if the available SNR supports that efficiency. A design must therefore specify its bandwidth, received power, noise conditions, and reliability requirement together.

The next four notebooks turn these ideas into a hands-on **acoustic communication playground**. Loudspeakers and microphones make transmission observable: messages become sounds that can be heard, recorded, and decoded. The propagation conditions differ from radio, but the same principles of modulation, bandwidth, noise, synchronization, and reliable reception apply.

Begin with [02.10 Acoustic Communication](02.10_Acoustic_Communication.ipynb), then explore [noise and detection](02.11_Acoustic_Noise_and_Detection.ipynb), [synchronization and framing](02.12_Acoustic_Synchronization_and_Framing.ipynb), and finally [a live link between two devices](02.13_Over_the_Air_Communication.ipynb). Together, these activities put the radio concepts into practice using sound as an accessible transmission medium.


---

### 9.4 Hands-On Exercises


<div class="alert alert-block alert-info">Fill in the responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>


The following three questions review the main ideas. Short answers are sufficient.

**Use the SNR slider in Section 9.2. As noise increases, which constellation is easier to distinguish: QPSK or 16-QAM? Briefly explain why.**

(answer here)


**A channel has a bandwidth of 5 MHz and an SNR of 10 dB. What is its Shannon capacity in Mbit/s?**

Hint: 10 dB corresponds to a linear SNR of 10. Use $C = B\log_2(1+\mathrm{SNR})$. Entering $B=5$ in MHz gives the result in Mbit/s.

(answer here)


**If the bandwidth doubles while the SNR stays the same, what happens to the Shannon capacity?**

(answer here)


**OPTIONAL: Have all tasks in this notebook been read and understood? Are there any shortcomings, missing parts, or sections that should be improved?**

Please help the teaching team improve the notebook by identifying unclear tasks or explanations and suggesting improvements.

(answer here)
